## Objective

The goal of this notebook is to predict how long an ITSM incident will take
to resolve using machine learning regression models.

Unlike classification, where we predict a Yes/No outcome,
regression predicts a continuous numeric value.

Here, the target variable is:
- `resolution_time_hours`

This helps with:
- Capacity planning
- Workload forecasting
- SLA buffer estimation

In [3]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Load Feature-Engineered Dataset

In [4]:

#We load the same feature-engineered dataset used in classification.
#The difference is only the target variable.

df = pd.read_csv("../data/processed/incident_ml_features.csv")
df.shape


(24918, 30)

Define Target Variable

In [5]:
#The regression target is the total resolution time of an incident
#measured in hours.

target = 'resolution_time_hours'

X = df.drop(columns=[target])
y = df[target]

Remove Target Leakage & Unsafe Features

Resolution time is known only after an incident is closed.
Any features derived from post-resolution information must be removed
to prevent target leakage.


In [6]:
leakage_cols = [
    'made_sla',
    'sla_breach_flag'
]

X = X.drop(columns=leakage_cols, errors='ignore')


Remove Non-Numeric Columns

Baseline regression models require numeric input.
All object and categorical columns are removed.

In [7]:
non_numeric_cols = X.select_dtypes(include=['object', 'category']).columns
X = X.drop(columns=non_numeric_cols)
X.dtypes

sys_mod_count                int64
impact                     float64
urgency                    float64
knowledge                     bool
u_priority_confirmation       bool
aging_days                   int64
priority                     int64
hour_of_day                  int64
day_of_week                  int64
is_weekend                   int64
location_freq              float64
dtype: object

Handle Missing Values

Regression models do not accept missing values.
Fully missing columns are removed and remaining NaNs are imputed using the median.


In [8]:
# Drop columns with all NaN values
all_nan_cols = X.columns[X.isnull().all()]
X = X.drop(columns=all_nan_cols)

# Fill remaining NaNs with median
for col in X.columns:
    if X[col].isnull().any():
        X[col] = X[col].fillna(X[col].median())

# Final check
X.isnull().sum().sum()


np.int64(0)

Time-Aware Train/Test Split

In [9]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]
